# Three-threshold interpretation and structural robustness

This notebook closes the three-threshold intuition phase by asking which conclusions survive changes in symbol, absolute threshold size, direction, and calendar period.

The cases are declared before any results are calculated. They are not searched or ranked for an attractive propagation rate. All symbols use their common date intersection so cross-symbol comparisons are not driven by different market periods.

This is structural robustness, not predictive out-of-sample validation. Terminal depth and propagated/rejected remain hindsight labels. Live checkpoint measurements use only information available on their observation bars.

In [1]:
from enum import StrEnum
from pathlib import Path
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import roc_auc_score

from dc_basics.core import dc_live_features, parse_dc_events

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
CHECKPOINT_BARS = [0, 5, 10, 20, 40, 80, 160, 320]
MIN_AUC_CLASS_COUNT = 20
SHORT_INTERPRETATION_CHECKPOINT = 20
LONG_INTERPRETATION_CHECKPOINT = 80

In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    REACHED_LARGE = "reached_large"
    AMBIGUOUS_SMALL_MEDIUM = "ambiguous_small_medium"
    AMBIGUOUS_MEDIUM_LARGE = "ambiguous_medium_large"
    UNRESOLVED_BELOW_MEDIUM = "unresolved_below_medium"
    UNRESOLVED_BELOW_LARGE = "unresolved_below_large"
    UNLINKED_AFTER_MEDIUM = "unlinked_after_medium"


class Stage(StrEnum):
    SMALL_TO_MEDIUM = "small_to_medium"
    MEDIUM_TO_LARGE = "medium_to_large"


class CaseId(StrEnum):
    GBP_15_30_45 = "GBPUSD 15/30/45"
    GBP_20_40_60 = "GBPUSD 20/40/60"
    GBP_25_50_75 = "GBPUSD 25/50/75"
    EUR_20_40_60 = "EURUSD 20/40/60"
    JPY_20_40_60 = "USDJPY 20/40/60"


class CaseSpec(NamedTuple):
    case_id: CaseId
    symbol: str
    small_theta: float
    medium_theta: float
    large_theta: float


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE)
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}
CASE_SPECS = (
    CaseSpec(CaseId.GBP_15_30_45, "GBPUSD.r", 0.0015, 0.0030, 0.0045),
    CaseSpec(CaseId.GBP_20_40_60, "GBPUSD.r", 0.0020, 0.0040, 0.0060),
    CaseSpec(CaseId.GBP_25_50_75, "GBPUSD.r", 0.0025, 0.0050, 0.0075),
    CaseSpec(CaseId.EUR_20_40_60, "EURUSD.r", 0.0020, 0.0040, 0.0060),
    CaseSpec(CaseId.JPY_20_40_60, "USDJPY.r", 0.0020, 0.0040, 0.0060),
)

case_design = pd.DataFrame.from_records([
    {
        "case": spec.case_id.value,
        "symbol": spec.symbol,
        "small_theta": spec.small_theta,
        "medium_theta": spec.medium_theta,
        "large_theta": spec.large_theta,
        "small_medium_ratio": spec.medium_theta / spec.small_theta,
        "medium_large_ratio": spec.large_theta / spec.medium_theta,
    }
    for spec in CASE_SPECS
]).set_index("case")
case_design.style.format({
    "small_theta": "{:.2%}",
    "medium_theta": "{:.2%}",
    "large_theta": "{:.2%}",
    "small_medium_ratio": "{:.2f}",
    "medium_large_ratio": "{:.2f}",
})

,symbol,small_theta,medium_theta,large_theta,small_medium_ratio,medium_large_ratio
case,,,,,,
GBPUSD 15/30/45,GBPUSD.r,0.15%,0.30%,0.45%,2.00,1.50
GBPUSD 20/40/60,GBPUSD.r,0.20%,0.40%,0.60%,2.00,1.50
GBPUSD 25/50/75,GBPUSD.r,0.25%,0.50%,0.75%,2.00,1.50
EURUSD 20/40/60,EURUSD.r,0.20%,0.40%,0.60%,2.00,1.50
USDJPY 20/40/60,USDJPY.r,0.20%,0.40%,0.60%,2.00,1.50


## Comparison contract

There are two distinct robustness questions:

1. Cross-symbol robustness keeps 20/40/60 bps fixed and changes GBPUSD, EURUSD, and USDJPY.
2. Absolute-scale robustness keeps GBPUSD and the 1:2:3 threshold shape fixed while using 15/30/45, 20/40/60, and 25/50/75 bps.

The tests do not optimize a ladder. A relationship is considered structurally credible only if its direction and economic meaning recur with adequate sample size. Exact probabilities are expected to change.

In [3]:
from dc_basics.data import load_close as _load_close

def load_close(symbol: str) -> pd.Series:
    return _load_close(DATA_ROOT, symbol)


symbols = tuple(dict.fromkeys(spec.symbol for spec in CASE_SPECS))
raw_prices = {symbol: load_close(symbol) for symbol in symbols}
common_start = max(close.index.min() for close in raw_prices.values())
common_end = min(close.index.max() for close in raw_prices.values())
if common_start >= common_end:
    raise ValueError("The selected symbols have no common data interval.")

prices_by_symbol = {
    symbol: close.loc[common_start:common_end].copy()
    for symbol, close in raw_prices.items()
}
coverage_summary = pd.DataFrame.from_records([
    {
        "symbol": symbol,
        "bars": len(close),
        "start": close.index.min(),
        "end": close.index.max(),
        "missing_close": int(close.isna().sum()),
    }
    for symbol, close in prices_by_symbol.items()
]).set_index("symbol")
if coverage_summary["missing_close"].gt(0).any():
    raise ValueError("At least one aligned close series contains missing values.")

print(f"Common comparison window: {common_start} to {common_end}")
coverage_summary.style.format({"bars": "{:,}"})

Common comparison window: 2023-05-22 14:00:00+02:00 to 2026-11-03 10:40:00+02:00


,bars,start,end,missing_close
symbol,,,,
GBPUSD.r,"257,387",2023-05-22 14:00:00+02:00,2026-11-03 10:40:00+02:00,0
EURUSD.r,"257,662",2023-05-22 14:00:00+02:00,2026-11-03 10:40:00+02:00,0
USDJPY.r,"257,481",2023-05-22 14:00:00+02:00,2026-11-03 10:40:00+02:00,0


## Reconstruct the three clocks and adjacent contests

In [4]:
from dc_basics.events import prepare_event_ledger


def build_clocks(close: pd.Series, spec: CaseSpec) -> tuple[dict, dict, pd.DataFrame]:
    settings = {
        Scale.SMALL: {"prefix": "S", "theta": spec.small_theta},
        Scale.MEDIUM: {"prefix": "M", "theta": spec.medium_theta},
        Scale.LARGE: {"prefix": "L", "theta": spec.large_theta},
    }
    event_ledgers = {}
    live_frames = {}
    mode_frame = pd.DataFrame(index=close.index)
    for scale in SCALE_ORDER:
        scale_settings = settings[scale]
        raw_events = parse_dc_events(prices=close, theta=scale_settings["theta"])
        ledger = prepare_event_ledger(
            raw_events, scale_settings["prefix"], scale, scale_settings["theta"]
        )
        live = dc_live_features(close, theta=scale_settings["theta"])
        event_ledgers[scale] = ledger
        live_frames[scale] = live
        mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
        mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
        mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
        event_lookup = ledger.set_index("dcc_idx")["event_id"]
        mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(event_lookup)
    return event_ledgers, live_frames, mode_frame

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


In [6]:
def select_broad_seeds(small_medium: pd.DataFrame, mode_frame: pd.DataFrame) -> pd.DataFrame:
    previous_modes = mode_frame[["small_mode", "medium_mode", "large_mode"]].shift(1)
    seeds = small_medium.copy()
    seed_times = pd.DatetimeIndex(seeds["start_time"])
    for scale in SCALE_ORDER:
        mode_column = f"{scale.value}_mode"
        seeds[f"prior_{scale.value}_mode"] = previous_modes.loc[seed_times, mode_column].to_numpy()
    seeds["broad_agreement_before_seed"] = (
        seeds["prior_small_mode"].eq(seeds["prior_medium_mode"])
        & seeds["prior_medium_mode"].eq(seeds["prior_large_mode"])
    )
    return seeds.loc[seeds["broad_agreement_before_seed"]].copy()


from dc_basics.lineage import build_three_scale_linkages


In [7]:
def select_strict_linkages(linkages: pd.DataFrame) -> pd.DataFrame:
    resolved = linkages.loc[linkages["terminal_depth"].notna()].copy()
    stage_one_resolution_position = resolved["terminal_position"].where(
        resolved["terminal_depth"].eq(0), resolved["medium_position"]
    )
    stage_one_has_live_bar = pd.to_numeric(stage_one_resolution_position).gt(resolved["seed_position"])
    stage_two_has_live_bar = (
        resolved["terminal_depth"].eq(0)
        | pd.to_numeric(resolved["terminal_position"]).gt(pd.to_numeric(resolved["medium_position"]))
    )
    strict = resolved.loc[
        resolved["terminal_label_hindsight_at_seed"]
        & ~resolved["small_medium_same_bar"]
        & ~resolved["medium_large_same_bar"]
        & stage_one_has_live_bar
        & stage_two_has_live_bar
    ].copy()
    if strict.empty:
        raise ValueError("No strict-clean linked lineages were found.")
    if not stage_one_has_live_bar.loc[strict.index].all():
        raise AssertionError("A strict-clean Stage-1 lineage has no pre-resolution bar.")
    strict_stage_two = strict.loc[strict["terminal_depth"].ge(1)]
    if not strict_stage_two["terminal_position"].gt(strict_stage_two["medium_position"]).all():
        raise AssertionError("A strict-clean Stage-2 lineage has no pre-resolution bar.")
    return strict


def build_case_structure(close: pd.Series, spec: CaseSpec) -> dict[str, object]:
    ledgers, live_frames, mode_frame = build_clocks(close, spec)
    small_medium = build_adjacent_attempts(mode_frame, Scale.SMALL, Scale.MEDIUM)
    medium_large = build_adjacent_attempts(mode_frame, Scale.MEDIUM, Scale.LARGE)
    broad_seeds = select_broad_seeds(small_medium, mode_frame)
    linkages = build_three_scale_linkages(broad_seeds, medium_large)
    strict_linkages = select_strict_linkages(linkages)
    return {
        "spec": spec,
        "prices": close,
        "event_ledgers": ledgers,
        "live_frames": live_frames,
        "mode_frame": mode_frame,
        "small_medium_attempts": small_medium,
        "medium_large_attempts": medium_large,
        "broad_seeds": broad_seeds,
        "linkages": linkages,
        "strict_linkages": strict_linkages,
    }

## Run the pre-registered cases

This is the expensive cell. Each case reconstructs three independent DC clocks, both adjacent contests, and the exact medium-event hand-off. No result from one case changes another case's configuration.

In [8]:
case_results = {}
for spec in CASE_SPECS:
    print(f"Building {spec.case_id.value}...")
    case_results[spec.case_id] = build_case_structure(prices_by_symbol[spec.symbol], spec)

print(f"Completed {len(case_results)} pre-registered cases.")

Building GBPUSD 15/30/45...
Building GBPUSD 20/40/60...


C:\Users\okmic\AppData\Local\Temp\ipykernel_7500\2990466614.py:3: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stage_one_resolution_position = resolved["terminal_position"].where(


Building GBPUSD 25/50/75...
Building EURUSD 20/40/60...
Building USDJPY 20/40/60...
Completed 5 pre-registered cases.


## Audit 1: event clocks and terminal depth

Changing an absolute threshold should change event frequency and episode duration. It need not preserve exact transition probabilities. The more important questions are whether the ladder remains coherent and whether linked depth retains the same economic interpretation.

In [9]:
clock_records = []
lineage_records = []
depth_records = []
for case_id, result in case_results.items():
    spec = result["spec"]
    ledgers = result["event_ledgers"]
    linkages = result["linkages"]
    strict = result["strict_linkages"]
    for scale in SCALE_ORDER:
        clock_records.append({
            "case": case_id.value,
            "scale": scale.value,
            "threshold": getattr(spec, f"{scale.value}_theta"),
            "events": len(ledgers[scale]),
            "median_t": ledgers[scale]["t"].median(),
            "median_r": ledgers[scale]["r"].median(),
        })

    reached_medium = strict["terminal_depth"].ge(1)
    reached_large = strict["terminal_depth"].eq(2)
    lineage_records.append({
        "case": case_id.value,
        "broad_seeds": len(linkages),
        "strict_lineages": len(strict),
        "same_bar_small_medium": int(linkages["small_medium_same_bar"].sum()),
        "same_bar_medium_large": int(linkages["medium_large_same_bar"].sum()),
        "p_reach_medium": reached_medium.mean(),
        "p_reach_large_given_medium": reached_large.sum() / reached_medium.sum(),
        "p_reach_large_from_seed": reached_large.mean(),
        "median_total_resolution_bars": (
            pd.to_numeric(strict["terminal_position"]) - strict["seed_position"]
        ).median(),
    })
    for depth, count in strict["terminal_depth"].value_counts().sort_index().items():
        depth_records.append({
            "case": case_id.value,
            "terminal_depth": int(depth),
            "lineages": int(count),
            "share": count / len(strict),
        })

clock_summary = pd.DataFrame.from_records(clock_records)
lineage_summary = pd.DataFrame.from_records(lineage_records).set_index("case")
depth_summary = pd.DataFrame.from_records(depth_records)

display(clock_summary.set_index(["case", "scale"]).style.format({
    "threshold": "{:.2%}",
    "events": "{:,}",
    "median_t": "{:.1f}",
    "median_r": "{:.3e}",
}))
display(lineage_summary.style.format({
    "broad_seeds": "{:,}",
    "strict_lineages": "{:,}",
    "same_bar_small_medium": "{:,}",
    "same_bar_medium_large": "{:,}",
    "p_reach_medium": "{:.2%}",
    "p_reach_large_given_medium": "{:.2%}",
    "p_reach_large_from_seed": "{:.2%}",
    "median_total_resolution_bars": "{:.1f}",
}))
display(depth_summary.pivot(index="case", columns="terminal_depth", values="share").style.format("{:.2%}"))

,broad_seeds,strict_lineages,same_bar_small_medium,same_bar_medium_large,p_reach_medium,p_reach_large_given_medium,p_reach_large_from_seed,median_total_resolution_bars
case,,,,,,,,
GBPUSD 15/30/45,"2,140","2,073",46,38,46.60%,64.49%,30.05%,37.0
GBPUSD 20/40/60,"1,273","1,230",36,15,46.42%,62.17%,28.86%,66.0
GBPUSD 25/50/75,889,862,23,5,43.85%,62.17%,27.26%,97.0
EURUSD 20/40/60,"1,156","1,118",28,14,46.78%,65.20%,30.50%,71.0
USDJPY 20/40/60,"1,892","1,777",95,57,44.79%,66.46%,29.77%,40.0


terminal_depth,0,1,2
case,,,
EURUSD 20/40/60,53.22%,16.28%,30.50%
GBPUSD 15/30/45,53.40%,16.55%,30.05%
GBPUSD 20/40/60,53.58%,17.56%,28.86%
GBPUSD 25/50/75,56.15%,16.59%,27.26%
USDJPY 20/40/60,55.21%,15.03%,29.77%


In [10]:
probability_figure = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Reach medium from seed",
        "Reach large given medium",
        "Reach large from seed",
    ],
)
probability_columns = [
    "p_reach_medium",
    "p_reach_large_given_medium",
    "p_reach_large_from_seed",
]
for column_number, column in enumerate(probability_columns, start=1):
    probability_figure.add_trace(go.Bar(
        x=lineage_summary.index,
        y=lineage_summary[column],
        name=column,
        showlegend=False,
        marker={"color": "#2563eb"},
        hovertemplate="case=%{x}<br>probability=%{y:.2%}<extra></extra>",
    ), row=1, col=column_number)
    probability_figure.update_yaxes(tickformat=".0%", row=1, col=column_number)
probability_figure.update_layout(
    title="Linked propagation probabilities by pre-registered case",
    template="plotly_white",
    height=500,
)
probability_figure.show()

## Audit 2: causal live checkpoints

For each strict-clean lineage, checkpoint k is retained only when the stage is still unresolved after k bars. Features are calculated from the observation bar and earlier prices. The later stage outcome is attached only for diagnostic comparison.

The checkpoint panel is deliberately sparse: it evaluates selected causal snapshots instead of materializing every active bar for every robustness case.

In [11]:
from dc_basics.live import build_causal_running_extreme


from dc_basics.live import calculate_target_progress


def stage_definition(lineage: object, stage: Stage) -> dict[str, object]:
    if stage == Stage.SMALL_TO_MEDIUM:
        propagated = int(lineage.terminal_depth) >= 1
        return {
            "lower_scale": Scale.SMALL,
            "upper_scale": Scale.MEDIUM,
            "upper_theta_name": "medium_theta",
            "start_position": int(lineage.seed_position),
            "resolution_position": (
                int(lineage.medium_position) if propagated else int(lineage.terminal_position)
            ),
            "future_stage_outcome": (
                AttemptOutcome.PROPAGATED.value if propagated else AttemptOutcome.REJECTED.value
            ),
            "stage_outcome_information_time": (
                lineage.medium_time if propagated else lineage.terminal_time
            ),
        }
    propagated = int(lineage.terminal_depth) == 2
    return {
        "lower_scale": Scale.MEDIUM,
        "upper_scale": Scale.LARGE,
        "upper_theta_name": "large_theta",
        "start_position": int(lineage.medium_position),
        "resolution_position": int(lineage.terminal_position),
        "future_stage_outcome": (
            AttemptOutcome.PROPAGATED.value if propagated else AttemptOutcome.REJECTED.value
        ),
        "stage_outcome_information_time": lineage.terminal_time,
    }

In [12]:
def build_case_checkpoints(result: dict[str, object]) -> pd.DataFrame:
    spec = result["spec"]
    close = result["prices"]
    live_frames = result["live_frames"]
    mode_frame = result["mode_frame"]
    strict = result["strict_linkages"]
    running_extremes = {
        scale: build_causal_running_extreme(close, live_frames[scale])
        for scale in SCALE_ORDER
    }
    small_dcc_flag = (
        mode_frame["small_upward_dcc"] | mode_frame["small_downward_dcc"]
    ).astype(np.int64)
    small_dcc_cumulative = small_dcc_flag.cumsum().to_numpy(dtype=np.int64)
    records = []

    for lineage in strict.itertuples(index=False):
        stages = [Stage.SMALL_TO_MEDIUM]
        if int(lineage.terminal_depth) >= 1:
            stages.append(Stage.MEDIUM_TO_LARGE)

        for stage in stages:
            definition = stage_definition(lineage, stage)
            start_position = definition["start_position"]
            resolution_position = definition["resolution_position"]
            lower_scale = definition["lower_scale"]
            upper_scale = definition["upper_scale"]
            upper_theta = getattr(spec, definition["upper_theta_name"])
            start_price = float(close.iloc[start_position])
            direction_sign = 1.0 if lineage.target_mode == DcMode.UP.value else -1.0

            for elapsed_bars in CHECKPOINT_BARS:
                observation_position = start_position + elapsed_bars
                if observation_position >= resolution_position:
                    continue
                path_positions = np.arange(start_position, observation_position + 1, dtype=np.int64)
                path_prices = close.iloc[path_positions].to_numpy(dtype=np.float64)
                path_extremes = running_extremes[upper_scale].iloc[path_positions].to_numpy(dtype=np.float64)
                path_progress = np.array([
                    calculate_target_progress(price, extreme, lineage.target_mode, upper_theta)
                    for price, extreme in zip(path_prices, path_extremes, strict=True)
                ])
                signed_excursion = (path_prices / start_price - 1.0) * direction_sign
                lower_live = live_frames[lower_scale].iloc[observation_position]
                upper_live = live_frames[upper_scale].iloc[observation_position]
                observation_time = close.index[observation_position]
                if observation_time >= definition["stage_outcome_information_time"]:
                    raise AssertionError("A checkpoint includes a stage outcome already known.")

                records.append({
                    "case": spec.case_id.value,
                    "symbol": spec.symbol,
                    "lineage_id": lineage.seed_small_event_id,
                    "stage": stage.value,
                    "elapsed_bars": elapsed_bars,
                    "observation_time": observation_time,
                    "target_mode": lineage.target_mode,
                    "lower_tmv_live": lower_live["tmv_current"],
                    "lower_t_live": lower_live["t_current"],
                    "lower_r_live": lower_live["rdc_current"],
                    "upper_tmv_live": upper_live["tmv_current"],
                    "upper_t_live": upper_live["t_current"],
                    "upper_r_live": upper_live["rdc_current"],
                    "target_progress": path_progress[-1],
                    "max_target_progress": np.max(path_progress),
                    "max_favorable_excursion_theta": np.max(signed_excursion) / getattr(
                        spec, f"{lower_scale.value}_theta"
                    ),
                    "max_adverse_excursion_theta": np.max(-signed_excursion.clip(max=0.0)) / getattr(
                        spec, f"{lower_scale.value}_theta"
                    ),
                    "nested_small_dccs": (
                        small_dcc_cumulative[observation_position]
                        - small_dcc_cumulative[start_position]
                    ),
                    "future_stage_outcome": definition["future_stage_outcome"],
                    "stage_outcome_information_time": definition["stage_outcome_information_time"],
                    "future_terminal_depth": int(lineage.terminal_depth),
                    "terminal_outcome_information_time": lineage.terminal_time,
                })

    checkpoints = pd.DataFrame.from_records(records)
    checkpoints["feature_information_time"] = checkpoints["observation_time"]
    checkpoints["stage_outcome_is_hindsight"] = checkpoints["observation_time"].lt(
        checkpoints["stage_outcome_information_time"]
    )
    checkpoints["terminal_outcome_is_hindsight"] = checkpoints["observation_time"].lt(
        checkpoints["terminal_outcome_information_time"]
    )
    if not checkpoints["stage_outcome_is_hindsight"].all():
        raise AssertionError("A checkpoint stage outcome is not hindsight.")
    if not checkpoints["terminal_outcome_is_hindsight"].all():
        raise AssertionError("A checkpoint terminal outcome is not hindsight.")
    return checkpoints

In [13]:
checkpoint_parts = []
for case_id, result in case_results.items():
    print(f"Checkpointing {case_id.value}...")
    checkpoint_parts.append(build_case_checkpoints(result))
checkpoint_panel = pd.concat(checkpoint_parts, ignore_index=True)

checkpoint_audit = checkpoint_panel.groupby(["case", "stage"], sort=True).agg(
    checkpoint_rows=("lineage_id", "size"),
    lineages=("lineage_id", "nunique"),
    maximum_progress=("target_progress", "max"),
    maximum_nested_small_dccs=("nested_small_dccs", "max"),
    all_stage_labels_hindsight=("stage_outcome_is_hindsight", "all"),
    all_terminal_labels_hindsight=("terminal_outcome_is_hindsight", "all"),
)
checkpoint_audit.style.format({
    "checkpoint_rows": "{:,}",
    "lineages": "{:,}",
    "maximum_progress": "{:.6f}",
    "maximum_nested_small_dccs": "{:.0f}",
})

Checkpointing GBPUSD 15/30/45...
Checkpointing GBPUSD 20/40/60...
Checkpointing GBPUSD 25/50/75...
Checkpointing EURUSD 20/40/60...
Checkpointing USDJPY 20/40/60...


## Audit 3: survival conditioning across cases

Later checkpoints contain fewer and increasingly selected attempts. A falling later-propagation rate among survivors is a population statement, not proof that elapsed time alone causes failure.

In [14]:
stage_attempt_counts = (
    checkpoint_panel.loc[checkpoint_panel["elapsed_bars"].eq(0)]
    .groupby(["case", "stage"])["lineage_id"]
    .nunique()
)
survival_summary = checkpoint_panel.groupby(["case", "stage", "elapsed_bars"], sort=True).agg(
    surviving_attempts=("lineage_id", "nunique"),
    later_propagation_rate=(
        "future_stage_outcome",
        lambda values: values.eq(AttemptOutcome.PROPAGATED.value).mean(),
    ),
    target_progress_median=("target_progress", "median"),
    lower_tmv_median=("lower_tmv_live", "median"),
    lower_r_median=("lower_r_live", "median"),
    nested_small_dccs_median=("nested_small_dccs", "median"),
).reset_index()
survival_summary["survivor_share"] = survival_summary.apply(
    lambda row: row["surviving_attempts"] / stage_attempt_counts[(row["case"], row["stage"])],
    axis=1,
)

interpretation_survival = survival_summary.loc[
    survival_summary["elapsed_bars"].isin([0, SHORT_INTERPRETATION_CHECKPOINT, LONG_INTERPRETATION_CHECKPOINT])
]
interpretation_survival.set_index(["case", "stage", "elapsed_bars"]).style.format({
    "surviving_attempts": "{:,}",
    "later_propagation_rate": "{:.2%}",
    "target_progress_median": "{:.2%}",
    "lower_tmv_median": "{:.3f}",
    "lower_r_median": "{:.3e}",
    "nested_small_dccs_median": "{:.1f}",
    "survivor_share": "{:.2%}",
})

In [15]:
survival_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[Stage.SMALL_TO_MEDIUM.value, Stage.MEDIUM_TO_LARGE.value],
)
case_colors = {
    CaseId.GBP_15_30_45.value: "#16a34a",
    CaseId.GBP_20_40_60.value: "#2563eb",
    CaseId.GBP_25_50_75.value: "#7c3aed",
    CaseId.EUR_20_40_60.value: "#f59e0b",
    CaseId.JPY_20_40_60.value: "#dc2626",
}
for column_number, stage in enumerate(Stage, start=1):
    stage_rows = survival_summary.loc[survival_summary["stage"].eq(stage.value)]
    for case, case_rows in stage_rows.groupby("case", sort=False):
        survival_figure.add_trace(go.Scatter(
            x=case_rows["elapsed_bars"],
            y=case_rows["later_propagation_rate"],
            mode="lines+markers",
            name=case,
            legendgroup=case,
            showlegend=column_number == 1,
            line={"color": case_colors[case], "width": 2},
            customdata=case_rows[["surviving_attempts", "survivor_share"]].to_numpy(),
            hovertemplate=(
                "case=%{fullData.name}<br>elapsed=%{x}<br>later propagation=%{y:.2%}<br>"
                "survivors=%{customdata[0]:,.0f}<br>survivor share=%{customdata[1]:.2%}<extra></extra>"
            ),
        ), row=1, col=column_number)
    survival_figure.update_yaxes(tickformat=".0%", row=1, col=column_number)
    survival_figure.update_xaxes(title_text="Elapsed bars", row=1, col=column_number)
survival_figure.update_layout(
    title="Later propagation among checkpoint survivors — diagnostic hindsight grouping",
    template="plotly_white",
    height=550,
    legend={"orientation": "h", "y": 1.13, "x": 0.0},
)
survival_figure.show()

## Audit 4: live-feature overlap

Orientation-free AUC is an in-sample univariate overlap diagnostic among attempts surviving to the checkpoint. It does not provide a probability forecast. Results with fewer than the minimum number of propagated or rejected survivors are marked unavailable.

In [16]:
LIVE_FEATURE_COLUMNS = [
    "lower_tmv_live",
    "lower_t_live",
    "lower_r_live",
    "target_progress",
    "max_target_progress",
    "max_favorable_excursion_theta",
    "max_adverse_excursion_theta",
    "nested_small_dccs",
]

overlap_records = []
group_columns = ["case", "stage", "elapsed_bars"]
for group_key, checkpoint in checkpoint_panel.groupby(group_columns, sort=True):
    target = checkpoint["future_stage_outcome"].eq(AttemptOutcome.PROPAGATED.value).astype(np.int8)
    propagated_count = int(target.sum())
    rejected_count = int(len(target) - propagated_count)
    for feature in LIVE_FEATURE_COLUMNS:
        values = checkpoint[feature].replace([np.inf, -np.inf], np.nan)
        valid = values.notna()
        feature_values = values.loc[valid]
        feature_target = target.loc[valid]
        enough_classes = (
            feature_target.sum() >= MIN_AUC_CLASS_COUNT
            and (len(feature_target) - feature_target.sum()) >= MIN_AUC_CLASS_COUNT
            and feature_values.nunique() >= 2
        )
        if enough_classes:
            raw_auc = roc_auc_score(feature_target, feature_values)
            orientation_free_auc = max(raw_auc, 1.0 - raw_auc)
            stronger_direction = "higher" if raw_auc >= 0.5 else "lower"
        else:
            raw_auc = np.nan
            orientation_free_auc = np.nan
            stronger_direction = "insufficient_sample"
        overlap_records.append({
            "case": group_key[0],
            "stage": group_key[1],
            "elapsed_bars": group_key[2],
            "surviving_attempts": len(checkpoint),
            "propagated_survivors": propagated_count,
            "rejected_survivors": rejected_count,
            "feature": feature,
            "raw_auc_for_higher_value": raw_auc,
            "stronger_value_direction": stronger_direction,
            "orientation_free_auc": orientation_free_auc,
        })

overlap_summary = pd.DataFrame.from_records(overlap_records)
selected_overlap = overlap_summary.loc[
    overlap_summary["elapsed_bars"].isin([
        0, SHORT_INTERPRETATION_CHECKPOINT, LONG_INTERPRETATION_CHECKPOINT
    ])
    & overlap_summary["feature"].isin(["target_progress", "lower_tmv_live", "lower_r_live"])
]
selected_overlap.pivot(
    index=["case", "stage", "elapsed_bars"],
    columns="feature",
    values="orientation_free_auc",
).style.format("{:.3f}")

In [17]:
progress_auc = overlap_summary.loc[overlap_summary["feature"].eq("target_progress")].copy()
auc_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[Stage.SMALL_TO_MEDIUM.value, Stage.MEDIUM_TO_LARGE.value],
)
for column_number, stage in enumerate(Stage, start=1):
    stage_rows = progress_auc.loc[progress_auc["stage"].eq(stage.value)]
    for case, case_rows in stage_rows.groupby("case", sort=False):
        auc_figure.add_trace(go.Scatter(
            x=case_rows["elapsed_bars"],
            y=case_rows["orientation_free_auc"],
            mode="lines+markers",
            name=case,
            legendgroup=case,
            showlegend=column_number == 1,
            line={"color": case_colors[case], "width": 2},
            customdata=case_rows[[
                "propagated_survivors", "rejected_survivors", "stronger_value_direction",
            ]].to_numpy(),
            hovertemplate=(
                "case=%{fullData.name}<br>elapsed=%{x}<br>AUC=%{y:.3f}<br>"
                "propagated=%{customdata[0]:,.0f}<br>rejected=%{customdata[1]:,.0f}<br>"
                "associated direction=%{customdata[2]}<extra></extra>"
            ),
        ), row=1, col=column_number)
    auc_figure.add_hline(
        y=0.5, line={"color": "#64748b", "dash": "dot"}, row=1, col=column_number
    )
    auc_figure.update_yaxes(range=[0.48, 1.0], row=1, col=column_number)
    auc_figure.update_xaxes(title_text="Elapsed bars", row=1, col=column_number)
auc_figure.update_layout(
    title="Causal target-progress separation by checkpoint — outcome color is hindsight",
    template="plotly_white",
    height=550,
    legend={"orientation": "h", "y": 1.13, "x": 0.0},
)
auc_figure.show()

## Audit 5: direction and calendar-period stability

These are diagnostic strata. A relationship concentrated in one direction or one calendar year is weaker evidence of a general structural property. Small groups should be read with caution.

In [18]:
direction_records = []
year_records = []
for case_id, result in case_results.items():
    strict = result["strict_linkages"].copy()
    strict["seed_year"] = strict["seed_time"].dt.year
    for target_mode, group in strict.groupby("target_mode", sort=True):
        direction_records.append({
            "case": case_id.value,
            "target_mode": target_mode,
            "lineages": len(group),
            "p_reach_medium": group["terminal_depth"].ge(1).mean(),
            "p_reach_large": group["terminal_depth"].eq(2).mean(),
        })
    for seed_year, group in strict.groupby("seed_year", sort=True):
        year_records.append({
            "case": case_id.value,
            "seed_year": int(seed_year),
            "lineages": len(group),
            "p_reach_medium": group["terminal_depth"].ge(1).mean(),
            "p_reach_large": group["terminal_depth"].eq(2).mean(),
        })

direction_summary = pd.DataFrame.from_records(direction_records)
year_summary = pd.DataFrame.from_records(year_records)
display(direction_summary.set_index(["case", "target_mode"]).style.format({
    "lineages": "{:,}",
    "p_reach_medium": "{:.2%}",
    "p_reach_large": "{:.2%}",
}))
display(year_summary.set_index(["case", "seed_year"]).style.format({
    "lineages": "{:,}",
    "p_reach_medium": "{:.2%}",
    "p_reach_large": "{:.2%}",
}))

## Audit 6: interpretation matrix

The matrix below records whether each adequately sampled case exhibits the same qualitative live-development relationship. It is deliberately narrow:

- progress separation at 20 bars should exceed initiation separation;
- the associated raw direction should be higher progress for later propagation;
- survivor composition at 80 bars is compared with initiation without treating the difference as causal.

A failed row is evidence against generality, not an instruction to discard or retune that case.

In [19]:
def auc_lookup(case: str, stage: str, elapsed_bars: int, feature: str) -> pd.Series:
    rows = overlap_summary.loc[
        overlap_summary["case"].eq(case)
        & overlap_summary["stage"].eq(stage)
        & overlap_summary["elapsed_bars"].eq(elapsed_bars)
        & overlap_summary["feature"].eq(feature)
    ]
    if len(rows) != 1:
        raise AssertionError("Expected exactly one overlap row.")
    return rows.iloc[0]


interpretation_records = []
for case in lineage_summary.index:
    for stage in Stage:
        baseline = auc_lookup(case, stage.value, 0, "target_progress")
        developed = auc_lookup(
            case, stage.value, SHORT_INTERPRETATION_CHECKPOINT, "target_progress"
        )
        survival_stage = survival_summary.loc[
            survival_summary["case"].eq(case)
            & survival_summary["stage"].eq(stage.value)
        ].set_index("elapsed_bars")
        baseline_rate = survival_stage.loc[0, "later_propagation_rate"]
        long_rate = (
            survival_stage.loc[LONG_INTERPRETATION_CHECKPOINT, "later_propagation_rate"]
            if LONG_INTERPRETATION_CHECKPOINT in survival_stage.index
            else np.nan
        )
        interpretation_records.append({
            "case": case,
            "stage": stage.value,
            "auc_at_initiation": baseline["orientation_free_auc"],
            "auc_at_20_bars": developed["orientation_free_auc"],
            "auc_change": (
                developed["orientation_free_auc"] - baseline["orientation_free_auc"]
            ),
            "progress_direction_at_20": developed["stronger_value_direction"],
            "propagation_rate_at_initiation": baseline_rate,
            "propagation_rate_at_80_bars": long_rate,
            "survivor_rate_change": long_rate - baseline_rate,
            "adequate_20_bar_sample": (
                developed["propagated_survivors"] >= MIN_AUC_CLASS_COUNT
                and developed["rejected_survivors"] >= MIN_AUC_CLASS_COUNT
            ),
        })

interpretation_matrix = pd.DataFrame.from_records(interpretation_records)
interpretation_matrix.set_index(["case", "stage"]).style.format({
    "auc_at_initiation": "{:.3f}",
    "auc_at_20_bars": "{:.3f}",
    "auc_change": "{:+.3f}",
    "propagation_rate_at_initiation": "{:.2%}",
    "propagation_rate_at_80_bars": "{:.2%}",
    "survivor_rate_change": "{:+.2%}",
})

In [20]:
robustness_summary = interpretation_matrix.groupby("stage", sort=True).agg(
    cases=("case", "size"),
    adequately_sampled_cases=("adequate_20_bar_sample", "sum"),
    cases_with_higher_20_bar_auc=("auc_change", lambda values: int(values.gt(0.0).sum())),
    cases_where_higher_progress_associates_with_propagation=(
        "progress_direction_at_20",
        lambda values: int(values.eq("higher").sum()),
    ),
    median_auc_change=("auc_change", "median"),
    cases_with_lower_80_bar_survivor_rate=(
        "survivor_rate_change",
        lambda values: int(values.lt(0.0).sum()),
    ),
    median_survivor_rate_change=("survivor_rate_change", "median"),
)
robustness_summary.style.format({
    "cases": "{:,}",
    "adequately_sampled_cases": "{:,}",
    "cases_with_higher_20_bar_auc": "{:,}",
    "cases_where_higher_progress_associates_with_propagation": "{:,}",
    "median_auc_change": "{:+.3f}",
    "cases_with_lower_80_bar_survivor_rate": "{:,}",
    "median_survivor_rate_change": "{:+.2%}",
})

,cases,adequately_sampled_cases,cases_with_higher_20_bar_auc,cases_where_higher_progress_associates_with_propagation,median_auc_change,cases_with_lower_80_bar_survivor_rate,median_survivor_rate_change
stage,,,,,,,
medium_to_large,5,5,5,5,+0.127,5,-13.58%
small_to_medium,5,5,5,5,+0.138,5,-4.25%


# Three-threshold robustness completion gate

The three-threshold phase is complete only when you can explain:

1. Why all symbols are restricted to one common date interval.
2. Why the five cases are declared before seeing their results.
3. Which conclusions concern structure and which numbers are case-specific.
4. Why event counts and durations must change with absolute threshold size.
5. Why the original small DCC remains the lineage ID through a medium hand-off.
6. Why attained depth is causal but terminal depth is hindsight until resolution.
7. Why checkpoint-zero progress is already positive.
8. Why outcome-conditioned AUC is diagnostic even though every feature value is causal.
9. Why later checkpoint cohorts cannot be compared as if they were the original population.
10. Whether higher live progress repeatedly associates with later propagation across cases.
11. Whether that association adds more information than completed initiation features did.
12. Whether any conclusion depends strongly on one direction, year, symbol, or threshold ladder.
13. Why structural robustness is not evidence of trading profitability.
14. Why a future classifier must be evaluated walk-forward and add information beyond direct geometry.

If these distinctions survive the sprint, the next phase is the four- and five-threshold hierarchy: agreement breadth, propagation depth, scale fragmentation, and firing density. Classification remains downstream of that intuition.